In [3]:
from pathlib import Path
import pandas as pd

In [4]:
# CONFIG
DATA_DIR = Path("data")

TTC_DIR = DATA_DIR / "ttc_bus"
WEATHER_DIR = DATA_DIR / "weather"
GTFS_DIR = DATA_DIR / "ttc_gtfs"
ROAD_DIR = DATA_DIR / "road_restrictions"

In [5]:
# LOAD DATA

# TTC Bus Delays
bus_frames = []

for file in TTC_DIR.iterdir():
    if file.suffix == ".csv":
        bus_frames.append(pd.read_csv(file))

    elif file.suffix == ".xlsx":
        bus_frames.append(pd.read_excel(file))

bus_df = pd.concat(bus_frames, ignore_index=True)

# Weather
weather_df = pd.read_csv(WEATHER_DIR / "toronto_weather_2022_2025.csv")

# Road Restrictions
road_df = pd.concat([pd.read_csv(file, skiprows=1) for file in ROAD_DIR.glob("*.csv")], ignore_index=True)

# GTFS
routes_df = pd.read_csv(GTFS_DIR / "routes.txt")
trips_df = pd.read_csv(GTFS_DIR / "trips.txt")
stops_df = pd.read_csv(GTFS_DIR / "stops.txt")
stop_times_df = pd.read_csv(GTFS_DIR / "stop_times.txt")

C:\Users\amras\AppData\Local\Temp\ipykernel_58324\2952638526.py:17: DtypeWarning: Columns (11,13,15,17,25) have mixed types. Specify dtype option on import or set low_memory=False.
  weather_df = pd.read_csv(


In [6]:
print("\nBus")
print(bus_df.shape)
print(bus_df.columns.tolist())

print("\nWeather")
print(weather_df.shape)
print(weather_df.columns.tolist())

print("\nRoad")
print(road_df.shape)
print(road_df.columns.tolist())

print("\nRoutes")
print(routes_df.shape)
print(routes_df.columns.tolist())

print("\nTrips")
print(trips_df.shape)
print(trips_df.columns.tolist())

print("\nStops")
print(stops_df.shape)
print(stops_df.columns.tolist())

print("\nStop Times")
print(stop_times_df.shape)
print(stop_times_df.columns.tolist())


Bus
(258702, 15)
['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle', '_id', 'Line', 'Station', 'Code', 'Bound']

Weather
(35064, 33)
['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID', 'Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Flag', 'Temp (°C)', 'Temp Flag', 'Dew Point Temp (°C)', 'Dew Point Temp Flag', 'Rel Hum (%)', 'Rel Hum Flag', 'Precip. Amount (mm)', 'Precip. Amount Flag', 'Wind Dir (10s deg)', 'Wind Dir Flag', 'Wind Spd (km/h)', 'Wind Spd Flag', 'Visibility (km)', 'Visibility Flag', 'Stn Press (kPa)', 'Stn Press Flag', 'Hmdx', 'Hmdx Flag', 'Wind Chill', 'Wind Chill Flag', 'Weather', 'year', 'month']

Road
(2788, 41)
['ID', 'Road', 'Name', 'District', 'Latitude', 'Longitude', 'RoadClass', 'Planned', 'SeverityOverride', 'Source', 'CreatedTime', 'LastUpdated', 'StartTime', 'EndTime', 'WorkPeriod', 'Expired', 'Signing', 'Notification', 'WorkEventType', 'Contractor', 'PermitType', 'Description', 'Speci

#### TTC Bus Delay

In [10]:
bus_df = bus_df[["Date", "Route", "Time", "Day", "Location", "Incident", "Min Delay", "Direction"]]

#### Weather

#### Road Restrictions

In [7]:
road_df = road_df[["Road", "District", "StartTime", "EndTime", "WorkEventType", "CurrImpact"]]

#### GTFS

In [9]:
routes_df = routes_df[["route_id", "route_short_name", "route_long_name"]]

In [18]:
#Route Mapping
# print(bus_df["Route"].head())
# print(routes_df["route_short_name"].head())
# print(routes_df["route_long_name"].head())

    route_id  route_short_name route_long_name
96       320               320           Yonge
    route_id  route_short_name route_long_name
99       325               325       Don Mills


In [19]:
bus_df["Route"] = bus_df["Route"].astype(str)

routes_df["route_short_name"] = (
    routes_df["route_short_name"]
    .astype(str)
)

bus_df = bus_df.merge(
    routes_df[
        [
            "route_id",
            "route_short_name",
            "route_long_name"
        ]
    ],
    left_on="Route",
    right_on="route_short_name",
    how="left"
)

print(
    bus_df[
        ["Route", "route_long_name"]
    ].head(10)
)

  Route route_long_name
0   320           Yonge
1   325       Don Mills
2   320           Yonge
3   320           Yonge
4   320           Yonge
5   363       Ossington
6    96          Wilson
7   320           Yonge
8   320           Yonge
9   300  Bloor-Danforth


In [20]:
import requests

BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

url = f"{BASE_URL}/api/3/action/package_show"
params = {"id": "road-restrictions"}

package = requests.get(url, params=params).json()

for r in package["result"]["resources"]:
    print("\nNAME:", r.get("name"))
    print("URL:", r.get("url"))
    print("FORMAT:", r.get("format"))


NAME: readme
URL: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/2265bfca-e845-4613-b341-70ee2ac73fbe/resource/0c97292b-0198-41ac-b0e3-1939cbbf8282/download/readme.xlsx
FORMAT: XLSX

NAME: Road Restrictions (Version 3) - JSON
URL: https://secure.toronto.ca/opendata/cart/road_restrictions/v3?format=json
FORMAT: JSON

NAME: Road Restrictions (Version 3) - CSV
URL: https://secure.toronto.ca/opendata/cart/road_restrictions/v3?format=csv
FORMAT: CSV

NAME: Road Restrictions (Version 3) - XML
URL: https://secure.toronto.ca/opendata/cart/road_restrictions/v3?format=xml&stream=n
FORMAT: XML
